In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from mlp_model.losses.bce_loss import BCELoss
from mlp_model.losses.cce_loss import CCELoss
from mlp_model.activations.relu import ReLU
from mlp_model.activations.tanh import Tanh
from mlp_model.activations.sigmoid import Sigmoid
import visualisation.classification_visualise as cl_vs
import classification_training_methods as cl_train

In [ ]:
data_kind = "simple" # simple, three_gauss
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/classification/train" / f"data.{data_kind}.train.{data_count}.csv"
test_filepath = Path(project_root) / "data/classification/test" / f"data.{data_kind}.test.{data_count}.csv"

In [ ]:
df = pd.read_csv(train_filepath)
df.head(5)

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
X = df.drop(["cls"], axis=1).to_numpy(dtype=float)
y = (df["cls"] - 1).to_numpy().reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, 
    stratify=y.reshape(-1), random_state=42)

In [ ]:
plt.figure(figsize=(8, 6))

for cls in np.unique(y):
    mask = y_train.reshape(-1) == cls

    plt.scatter(
        X_train[mask, 0],
        X_train[mask, 1],
        label=f"Class {cls}",
        alpha=0.7,
        s=20
    )

In [ ]:
df_test = pd.read_csv(test_filepath)

X_test = df.drop(["cls"], axis=1).to_numpy(dtype=float)
y_test = (df["cls"] - 1).to_numpy().reshape(-1, 1)

## Wpływ liczb warstw ukrytych

In [ ]:
hidden = {
    "0 hidden": (),
    "1 hidden": (16,),
    "2 hidden": (16, 16,),
    "3 hidden": (16, 16, 16,),
    "4 hidden": (16, 16, 16, 16,),
}

trainings = {}

for name, hidden_sizes in hidden.items():
    trainings[name] = cl_train.run_training(hidden_sizes=hidden_sizes, X_train=X_train, y_train=y_train, X_val=X_val,
                                            y_val=y_val, activation_hidden=Sigmoid(), epochs=50,
                                            loss_cls=BCELoss(), learning_rate=0.01, classes_count=2)

    cl_vs.plot_decision_boundary(trainings[name][0]["trainer"], X_val, y_val, name)

cl_vs.plot_loss(trainings, "train_loss")
cl_vs.plot_loss(trainings, "val_loss")

display(cl_train.summarize_results(trainings))

In [ ]:
cl_vs.plot_weights(trainings["0 hidden"][0]["history"], layer_idx=0, n_weights=2, type="0 hidden")

In [ ]:
cl_vs.plot_weights(trainings["1 hidden"][0]["history"], layer_idx=1, n_weights=16, type="1 hidden")

In [ ]:
cl_vs.plot_weights(trainings["2 hidden"][0]["history"], layer_idx=1, n_weights=16, type="2 hidden")

In [ ]:
cl_vs.plot_weights(trainings["3 hidden"][0]["history"], layer_idx=1, n_weights=16, type="3 hidden")

In [ ]:
cl_vs.plot_weights(trainings["4 hidden"][0]["history"], layer_idx=1, n_weights=16, type="4 hidden")

In [ ]:
history = trainings["4 hidden"][0]["history"]

grad_norms = np.asarray(history["grad_norms"])

for layer_idx in range(grad_norms.shape[1]):
    print(
        f"Layer {layer_idx + 1}: "
        f"mean grad norm = {np.mean(grad_norms[:, layer_idx]):.12e}"
    )

In [ ]:
test_results = cl_train.evaluate_runs(trainings["0 hidden"], X_test, y_test)
print("0 hidden test results: ")
display(test_results)
print("0 hidden average test results: ")
display(test_results[["test_loss", "accuracy", "precision", "f1_score"]
].agg(["mean", "std"]))

In [ ]:
test_results = cl_train.evaluate_runs(trainings["4 hidden"], X_test, y_test)
print("4 hidden test results: ")
display(test_results)
print("4 hidden average test results: ")
display(test_results[["test_loss", "accuracy", "precision", "f1_score"]
].agg(["mean", "std"]))